# Load and plot DSA-03 full-band continuum (FC)

Load the forward-modeled DSA full-band continuum mosaic `mockfm_DSA-03_fc37_1348_20261002_restored.fits` (Jy/beam), display a section of it with `imshow`, and overlay celestial coordinates from the FITS header via `astropy.wcs`.

The mosaic is large, so the file is memory-mapped and only the requested section is read into memory.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.wcs import WCS

FILENAME = "mockfm_DSA-03_fc37_1348_20261002_restored.fits"

# JupyterHub: the DSA release is on the read-only S3 mount. DSA_DATA overrides it.
DS = Path(os.environ.get("DSA_DATA", "~/s3/schmidt-observatory-system/dsa/v2-convening")).expanduser()
matches = sorted(DS.rglob(FILENAME))
if not matches:
    raise FileNotFoundError(f"{FILENAME} not found under {DS}; set DSA_DATA to the directory containing it")
fits_path = matches[0]
print(f"Using: {fits_path}")

In [ ]:
with fits.open(fits_path, memmap=True) as hdul:
    hdul.info()
    header = hdul[0].header
    nx, ny = header["NAXIS1"], header["NAXIS2"]
    print(f"image size: {nx} x {ny} pixels (x, y)")
    print(f"BUNIT: {header.get('BUNIT', 'n/a')}   OBJECT: {header.get('OBJECT', 'n/a')}")

# Celestial WCS from the header (drops any degenerate frequency/Stokes axes)
wcs = WCS(header).celestial
print(wcs)

## Choose a section

Set the section as a centre pixel and a size in pixels. By default, a 2000 x 2000 pixel box at the image centre is shown.

In [ ]:
SIZE = 2000                      # section width/height in pixels
XC, YC = nx // 2, ny // 2        # centre pixel (x, y); change to look elsewhere

x0, x1 = max(XC - SIZE // 2, 0), min(XC + SIZE // 2, nx)
y0, y1 = max(YC - SIZE // 2, 0), min(YC + SIZE // 2, ny)

with fits.open(fits_path, memmap=True) as hdul:
    data = hdul[0].data
    # Leading singleton axes (Stokes/frequency) are indexed away; the last two are (y, x)
    lead = (0,) * (data.ndim - 2)
    section = np.array(data[lead + (slice(y0, y1), slice(x0, x1))], dtype=float)

sub_wcs = wcs[y0:y1, x0:x1]
print(f"section shape: {section.shape}; median {np.nanmedian(section):.3g}, rms {np.nanstd(section):.3g}")

In [ ]:
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(1, 1, 1, projection=sub_wcs)

finite = section[np.isfinite(section)]
norm = simple_norm(finite, "asinh", percent=99.5)
im = ax.imshow(section, origin="lower", cmap="gray", norm=norm)

ax.coords.grid(True, color="cyan", alpha=0.5, linestyle=":")
ax.set_xlabel("Right Ascension")
ax.set_ylabel("Declination")
ax.set_title(f"DSA-03 full-band continuum\n{fits_path.name}", fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(header.get("BUNIT", "pixel value"))
plt.tight_layout()
plt.show()